[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_05/MFM_ch5_mf2_garch/MFM_ch5_mf2_garch.ipynb)

# MFM_ch5_mf2_garch

Lecture case study (Conrad and Engle, 2025, Journal of Applied Econometrics): the MF2-GARCH (multiplicative factor multi-frequency GARCH) with a short-term GJR-GARCH(1,1) component and a long-term component driven by the m-day average of the short-term component's standardised forecast errors, estimated by Gaussian quasi-maximum likelihood on daily S&P 500 log returns, 3 January 1990 to 18 September 2026; the window m chosen by the BIC (m = 5 to 160), Bollerslev-Wooldridge robust standard errors, the conditional and long-term volatility, and out-of-sample forecasts 2010-2026 (rolling 20-year window, re-estimated every 21 trading days) for one day, one week and the forward variance of months 1 to 8, evaluated by QLIKE and RMSE relative to the historical forecast and by Diebold-Mariano tests against the GJR-GARCH.

*Modelling Financial Markets (MFM), Chapter 5: Conditional Volatility, GARCH Models. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your Bucharest University of Economic Studies (ASE) Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (comma-separated values (CSV), charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_5'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


Save your outputs to: /content/mfm-audit/20261016T163406-c43bff


## Data

In [ ]:
import os
import re
import urllib.request
import numpy as np
import pandas as pd

# local copy of the course data, if the notebook runs inside the repository
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# name -> (symbol, label, group, start date)
MARKETS = {
    'sp500':  ('GSPC.INDX',      'S&P 500',                 'Equity', '2000-01-01'),
    'bet':    ('BET',            'BET',                     'Equity', '2000-01-01'),
    'bettr':  ('BETTR.INDX',     'BET-TR',                  'Equity', '2014-09-23'),
    'btc':    ('BTC-USD.CC',     'Bitcoin',                 'Crypto', '2014-09-17'),
    'eurron': ('REF:EUR',        'EUR/RON (BNR reference)', 'FX',     '2005-07-01'),
    'gold':   ('XAUUSD.FOREX',   'Gold (XAU/USD)',          'Commodity', '2000-01-01'),
}
LABELS = {k: v[1] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Daily price table of one symbol (local copy of the course data, or GitHub)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def load_close(name, start=None, end=END):
    """Daily closing price, cleaned with the chapter conventions."""
    symbol, _, group, start0 = MARKETS[name]
    start = start or start0
    d = read_market(symbol).loc[start:end]
    d = d[d['close'] > 0].dropna(subset=['close'])
    if group != 'Crypto':
        d = d[d.index.dayofweek < 5]          # no weekend quotes
        d = d[~artificial_fill(d)]            # non-trading days filled with the previous price
    return d['close'].rename(name)


def artificial_fill(d):
    """Rows that repeat the previous close and show no trading: an artificial non-trading-day fill.

    A repeated close is a fill only if the row itself shows no trading: zero volume (for files that report
    volume), no intraday range (open = high = low = close) or a copy of the previous row in every field. A
    repeated close on a day with volume and an intraday range is a genuine unchanged close (a zero return)
    and is kept. Files with closes only (the BET) give no such evidence, so every row is kept.
    """
    repeat = d['close'].diff() == 0
    no_trade = pd.Series(False, index=d.index)
    if 'volume' in d and (d['volume'] > 0).mean() > 0.5:
        no_trade |= d['volume'].fillna(0) <= 0
    if {'open', 'high', 'low'} <= set(d.columns):
        no_trade |= (d['open'] == d['high']) & (d['high'] == d['low']) & (d['low'] == d['close'])
        fields = [c for c in ['open', 'high', 'low', 'close', 'volume'] if c in d]
        no_trade |= (d[fields] == d[fields].shift()).all(axis=1)
    return repeat & no_trade


def pct_returns(name, start=None, end=END):
    """Daily log returns in percent, on the series' own calendar."""
    return (100 * np.log(load_close(name, start, end)).diff().dropna()).rename(name)


def periods_per_year(r):
    """Average number of observations per calendar year (actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years




## Model, estimation and forecast evaluation

MF2-GARCH-rw-m (equations 4-6 of the paper) and the nested GJR-GARCH(1,1), Gaussian QMLE; closed-form multi-step forecasts of the conditional variance.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import optimize
from numba import njit
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 11.5
plt.rcParams['ytick.labelsize'] = 11.5
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.4
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 11

MainBlue, IDAred, Forest, Navy, BandBlue = '#1A3A6E', '#CD0000', '#2E7D32', '#1F2A44', '#C5D2E8'
CHART_DIR = OUT_DIR
OUT_JSON = os.path.join(OUT_DIR, 'ch5_mf2_case.json')

START, END = '1990-01-01', '2026-09-18'
M_GRID = list(range(5, 161))
BURN = 160                      # likelihood summed from the same day for every m
OOS_START = '2010-01-01'
STEP = 21                       # re-estimation every 21 trading days
HOR = [('Day', 1, 1), ('Week', 1, 5)] + [(f'{k}m', 21 * (k - 1) + 1, 21 * k) for k in range(1, 9)]
SMAX = 168
NBER = [('1990-07-01', '1991-03-31'), ('2001-03-01', '2001-11-30'), ('2007-12-01', '2009-06-30'),
        ('2020-02-01', '2020-04-30')]
# the paper's numbers (Table 2: January 1971 - June 2023; Table 4: out of sample 2010-2019)
PAPER = dict(gjr=dict(alpha=0.023, gamma=0.115, beta=0.901, phi=0.982),
             mf2=dict(m=63, alpha=0.003, gamma=0.162, beta=0.840, lambda0=0.018, lambda1=0.112, lambda2=0.870,
                      phi=0.924, lam=0.982),
             t4_rmse=dict(gjr={'1m': 0.798, '8m': 0.866}, mf2={'1m': 0.685, '8m': 0.811}),
             t4_qlike=dict(gjr={'1m': 0.723, '8m': 0.918}, mf2={'1m': 0.708, '8m': 0.859}))


# ----------------------------------------------------------------------------- filters (numba)
@njit
def gjr_filter(e, omega, alpha, gamma, beta, h0):
    n = e.shape[0]
    h = np.empty(n + 1)
    h[0] = h0
    for t in range(1, n + 1):
        a = alpha + (gamma if e[t - 1] < 0 else 0.0)
        h[t] = omega + a * e[t - 1] ** 2 + beta * h[t - 1]
    return h            # h[t]: variance of e[t]; h[n]: one-day-ahead forecast after the last day


@njit
def mf2_filter(e, alpha, gamma, beta, l0, l1, l2, m, tau0):
    """h, tau and V = e^2/h for t = 0..n (index n: the values for the day after the last observation)."""
    n = e.shape[0]
    phi = alpha + gamma / 2 + beta
    h = np.empty(n + 1)
    tau = np.empty(n + 1)
    V = np.empty(n)
    h[0] = 1.0
    tau[0] = tau0
    run = 0.0
    for t in range(n):
        V[t] = e[t] ** 2 / h[t]
        run += V[t]
        if t >= m:
            run -= V[t - m]
        a = alpha + (gamma if e[t] < 0 else 0.0)
        h[t + 1] = (1 - phi) + a * e[t] ** 2 / tau[t] + beta * h[t]
        if t + 1 >= m:
            tau[t + 1] = l0 + l1 * run / m + l2 * tau[t]
        else:
            tau[t + 1] = tau0                     # the first m days: tau fixed at the sample variance
    return h, tau, V


@njit
def mf2_forecast(V_last, h1, tau1, alpha, gamma, beta, l0, l1, l2, m, kappa, S):
    """E_t[sigma^2_{t+s}], s = 1..S, given h_{t+1}, tau_{t+1} and the last m values V_{t-m+1..t} (oldest first).

    Derived from the recursions: with a_k = (alpha + gamma 1{Z_k<0}) Z_k^2 + beta, E a = phi and
    E[a Z^2] = phi_kappa = (alpha + gamma/2) kappa + beta for a symmetric density (Assumption 1 of the paper):
      E[h_{t+s} tau_{t+s-1}] = (1-phi) E tau_{t+s-1} + phi E sigma^2_{t+s-1}
      E[h_{t+s} V_{k}]       = (1-phi) (1 + phi + ... + phi^{j-1}) E tau_k + phi^{j-1} phi_kappa E sigma^2_k, k = t+s-j > t
      E[h_{t+s} V_{k}]       = V_k E h_{t+s}, k <= t
    """
    phi = alpha + gamma / 2 + beta
    phik = (alpha + gamma / 2) * kappa + beta
    Eh = np.empty(S + 1)
    Et = np.empty(S + 1)
    Es = np.empty(S + 1)
    Eh[1] = h1
    Et[1] = tau1
    Es[1] = h1 * tau1
    for s in range(2, S + 1):
        Eh[s] = 1 + phi ** (s - 1) * (h1 - 1)
        # E tau_{t+s}
        acc = 0.0
        for j in range(1, m + 1):
            k = s - j                      # k >= 1: future day t+k; k <= 0: known V_{t+k}
            acc += Et[k] if k >= 1 else V_last[m - 1 + k]
        Et[s] = l0 + l1 * acc / m + l2 * Et[s - 1]
        # E sigma^2_{t+s}
        acc = 0.0
        for j in range(1, m + 1):
            k = s - j
            if k >= 1:
                geo = (1 - phi ** j) / (1 - phi)
                acc += (1 - phi) * geo * Et[k] + phi ** (j - 1) * phik * Es[k]
            else:
                acc += V_last[m - 1 + k] * Eh[s]
        Es[s] = l0 * Eh[s] + l2 * ((1 - phi) * Et[s - 1] + phi * Es[s - 1]) + l1 * acc / m
    return Es[1:]


# ----------------------------------------------------------------------------- likelihoods
def _ll_obs(s2, e):
    return -0.5 * (np.log(2 * np.pi) + np.log(s2) + e ** 2 / s2)


def gjr_llobs(theta, r, burn):
    mu, omega, alpha, gamma, beta = theta
    e = r - mu
    h = gjr_filter(e, omega, alpha, gamma, beta, np.var(r))[:-1]
    return _ll_obs(h[burn:], e[burn:])


def mf2_llobs(theta, r, m, burn):
    mu, alpha, gamma, beta, l0, l1, l2 = theta
    e = r - mu
    h, tau, _ = mf2_filter(e, alpha, gamma, beta, l0, l1, l2, m, np.var(r))
    return _ll_obs((h * tau)[:-1][burn:], e[burn:])


def _neg(f, *a):
    def g(theta):
        v = f(theta, *a).sum()
        return 1e10 if not np.isfinite(v) else -v
    return g


def fit_gjr(r, burn, x0=None):
    x0 = np.array(x0 if x0 is not None else [r.mean(), 0.02, 0.02, 0.12, 0.90])
    cons = [{'type': 'ineq', 'fun': lambda x: 0.9999 - x[2] - x[3] / 2 - x[4]}]
    b = [(-1, 1), (1e-6, 2), (0, 0.5), (0, 0.8), (0, 0.9999)]
    res = optimize.minimize(_neg(gjr_llobs, r, burn), x0, method='SLSQP', bounds=b, constraints=cons,
                            options=dict(maxiter=500, ftol=1e-10))
    return res.x, -res.fun


def fit_mf2(r, m, burn, x0=None):
    x0 = np.array(x0 if x0 is not None else [r.mean(), 0.01, 0.15, 0.84, 0.02, 0.10, 0.87])
    x0[4] = max(x0[4], 1e-4)
    cons = [{'type': 'ineq', 'fun': lambda x: 0.9999 - x[1] - x[2] / 2 - x[3]},
            {'type': 'ineq', 'fun': lambda x: 0.9999 - x[5] - x[6]}]
    b = [(-1, 1), (0, 0.5), (0, 0.8), (0, 0.9999), (1e-6, 5), (0, 1), (0, 0.9999)]
    res = optimize.minimize(_neg(mf2_llobs, r, m, burn), x0, method='SLSQP', bounds=b, constraints=cons,
                            options=dict(maxiter=500, ftol=1e-10))
    return res.x, -res.fun


def robust_se(llobs, theta, *a):
    """Bollerslev-Wooldridge sandwich: H^{-1} J H^{-1}, numerical scores and Hessian."""
    k = len(theta)
    eps = 1e-5 * np.maximum(np.abs(theta), 1e-2)
    G = np.empty((len(llobs(theta, *a)), k))
    for i in range(k):
        d = np.zeros(k); d[i] = eps[i]
        G[:, i] = (llobs(theta + d, *a) - llobs(theta - d, *a)) / (2 * eps[i])
    H = np.empty((k, k))
    for i in range(k):
        d = np.zeros(k); d[i] = eps[i]
        gp = np.array([((llobs(theta + d + dd, *a) - llobs(theta + d - dd, *a)).sum()) / (2 * eps[j])
                       for j, dd in ((j, np.eye(k)[j] * eps[j]) for j in range(k))])
        gm = np.array([((llobs(theta - d + dd, *a) - llobs(theta - d - dd, *a)).sum()) / (2 * eps[j])
                       for j, dd in ((j, np.eye(k)[j] * eps[j]) for j in range(k))])
        H[:, i] = (gp - gm) / (2 * eps[i])
    H = (H + H.T) / 2
    Hi = np.linalg.inv(-H)
    return np.sqrt(np.diag(Hi @ (G.T @ G) @ Hi))


# ----------------------------------------------------------------------------- full sample
def full_sample(r):
    x = r.values
    n_eff = len(x) - BURN
    th_g, ll_g = fit_gjr(x, BURN)
    bic_g = (-2 * ll_g + 5 * np.log(n_eff)) / n_eff
    rows, x0 = [], None
    for m in M_GRID:
        th, ll = fit_mf2(x, m, BURN, x0)
        if x0 is None or ll > -1e9:
            x0 = th
        rows.append(dict(m=m, ll=ll, bic=(-2 * ll + 7 * np.log(n_eff)) / n_eff, theta=th))
    # second pass from the neighbours' estimates, to avoid local optima along the grid
    for i in range(len(rows)):
        for j in (i - 1, i + 1):
            if 0 <= j < len(rows):
                th, ll = fit_mf2(x, rows[i]['m'], BURN, rows[j]['theta'].copy())
                if ll > rows[i]['ll'] + 1e-6:
                    rows[i].update(ll=ll, bic=(-2 * ll + 7 * np.log(n_eff)) / n_eff, theta=th)
    best = min(rows, key=lambda d: d['bic'])
    m = best['m']
    th = best['theta']
    se = robust_se(mf2_llobs, th, x, m, BURN)
    se_g = robust_se(gjr_llobs, th_g, x, BURN)
    e = x - th[0]
    h, tau, V = mf2_filter(e, *th[1:], m, np.var(x))
    z = e / np.sqrt(h[:-1] * tau[:-1])
    out = dict(n=len(x), n_eff=n_eff, start=str(r.index[0].date()), end=str(r.index[-1].date()), m=m,
               gjr=dict(zip(['mu', 'omega', 'alpha', 'gamma', 'beta'], th_g.tolist())),
               gjr_se=dict(zip(['mu', 'omega', 'alpha', 'gamma', 'beta'], se_g.tolist())),
               mf2=dict(zip(['mu', 'alpha', 'gamma', 'beta', 'lambda0', 'lambda1', 'lambda2'], th.tolist())),
               mf2_se=dict(zip(['mu', 'alpha', 'gamma', 'beta', 'lambda0', 'lambda1', 'lambda2'], se.tolist())),
               ll_gjr=ll_g, ll_mf2=best['ll'], bic_gjr=bic_g, bic_mf2=best['bic'],
               bic_grid=[dict(m=d['m'], bic=d['bic'], ll=d['ll']) for d in rows],
               grid_theta={str(d['m']): d['theta'].tolist() for d in rows},
               kappa=float(np.mean(z[BURN:] ** 4)))
    p = out['gjr']
    out['gjr']['phi'] = p['alpha'] + p['gamma'] / 2 + p['beta']
    q = out['mf2']
    out['mf2']['phi'] = q['alpha'] + q['gamma'] / 2 + q['beta']
    out['mf2']['lam'] = q['lambda1'] + q['lambda2']
    for mm in (21, 63, 126):
        d = next(d for d in rows if d['m'] == mm)
        out[f'bic_m{mm}'] = d['bic']
    series = pd.DataFrame({'sigma': np.sqrt(h[:-1] * tau[:-1]), 'tau': np.sqrt(tau[:-1])}, index=r.index)
    return out, series


# ----------------------------------------------------------------------------- out of sample
def out_of_sample(r):
    x = r.values
    idx = r.index
    i0 = int(np.searchsorted(idx, pd.Timestamp(OOS_START)))
    W = i0                                            # rolling window: 20 years (1990-2009)
    # m chosen once, by the BIC on the first window
    xw = x[:W]
    nw = W - BURN
    best, x0 = None, None
    for m in M_GRID:
        th, ll = fit_mf2(xw, m, BURN, x0)
        x0 = th
        b = (-2 * ll + 7 * np.log(nw)) / nw
        if best is None or b < best[0]:
            best = (b, m)
    m = best[1]
    n = len(x)
    last_origin = n - 1                               # origins: days i0-1 .. n-2 (forecast for day i0 onwards)
    F = {k: np.full((n, len(HOR)), np.nan) for k in ('gjr', 'mf2', 'hist')}
    th_m, th_g = None, None
    for b0 in range(i0, n, STEP):
        lo = b0 - W
        win = x[lo:b0]
        th_m, _ = fit_mf2(win, m, BURN, th_m)
        th_g, _ = fit_gjr(win, BURN, th_g)
        b1 = min(b0 + STEP, n)
        seg = x[lo:b1]
        # MF2: filter with the parameters estimated on the window ending at b0-1
        e = seg - th_m[0]
        h, tau, V = mf2_filter(e, *th_m[1:], m, np.var(win))
        zz = (e / np.sqrt(h[:-1] * tau[:-1]))[BURN:W]
        kappa = float(np.mean(zz ** 4))
        eg = seg - th_g[0]
        hg = gjr_filter(eg, *th_g[1:], np.var(win))
        om, al, ga, be = th_g[1:]
        pg = al + ga / 2 + be
        ug = om / (1 - pg)
        for t in range(b0 - 1, b1 - 1):              # origin t (close of day t), forecasts for t+1, ...
            j = t - lo                                # position in seg
            fm = mf2_forecast(V[j - m + 1:j + 1].copy(), h[j + 1], tau[j + 1], *th_m[1:], m, kappa, SMAX)
            s = np.arange(1, SMAX + 1)
            fg = ug + pg ** (s - 1) * (hg[j + 1] - ug)
            hist = np.mean(x[max(0, t - 2519):t + 1] ** 2)
            for c, (_, a, bb) in enumerate(HOR):
                F['mf2'][t, c] = fm[a - 1:bb].sum()
                F['gjr'][t, c] = fg[a - 1:bb].sum()
                F['hist'][t, c] = hist * (bb - a + 1)
    # realised proxies: sums of squared returns over the same days
    r2 = x ** 2
    cs = np.concatenate([[0], np.cumsum(r2)])
    Y = np.full((n, len(HOR)), np.nan)
    for c, (_, a, bb) in enumerate(HOR):
        for t in range(i0 - 1, n):
            if t + bb < n:
                Y[t, c] = cs[t + bb + 1] - cs[t + a]
    return dict(m=m, F=F, Y=Y, idx=idx, i0=i0)


def evaluate(o, end=None):
    idx = o['idx']
    res = {}
    for c, (lab, a, bb) in enumerate(HOR):
        ok = ~np.isnan(o['Y'][:, c]) & ~np.isnan(o['F']['mf2'][:, c])
        if end is not None:
            ok &= np.asarray(idx + pd.Timedelta(0) <= pd.Timestamp(end))
            # the target window must end before the cut-off as well
            last = np.searchsorted(idx, pd.Timestamp(end), side='right') - 1
            ok &= np.arange(len(idx)) + bb <= last
        y = o['Y'][ok, c]
        row = {}
        L = {}
        for k in ('hist', 'gjr', 'mf2'):
            f = o['F'][k][ok, c]
            pos = y > 0
            ql = np.full_like(y, np.nan)
            ql[pos] = y[pos] / f[pos] - np.log(y[pos] / f[pos]) - 1
            L[k] = dict(se=(y - f) ** 2, ql=ql)
            row[k] = dict(rmse=float(np.sqrt(np.mean((y - f) ** 2))), qlike=float(np.nanmean(ql)))
        for k in ('gjr', 'mf2'):
            row[k]['rmse_rel'] = row[k]['rmse'] / row['hist']['rmse']
            row[k]['qlike_rel'] = row[k]['qlike'] / row['hist']['qlike']
        # Diebold-Mariano, MF2 minus GJR, Newey-West with as many lags as the overlap of the targets
        for loss in ('ql', 'se'):
            d = L['mf2'][loss] - L['gjr'][loss]
            d = d[~np.isnan(d)]
            ols = sm.OLS(d, np.ones_like(d)).fit(cov_type='HAC', cov_kwds={'maxlags': int(bb)})
            row[f'dm_{loss}'] = float(ols.tvalues[0])
        row['n'] = int(ok.sum())
        row['zero_days'] = int((y <= 0).sum())
        res[lab] = row
    return res


def fig_case(series, oos_full, oos_pre, ppy, m):
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11, 3.9), gridspec_kw=dict(width_ratios=[1.55, 1]))
    A = np.sqrt(ppy)
    for k, (a, b) in enumerate(NBER):
        ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color=BandBlue, alpha=0.7, lw=0,
                   label='NBER recessions' if k == 0 else None)
    ax.plot(series.index, series['sigma'] * A, color=MainBlue, lw=0.5, label=r'Conditional volatility $\sqrt{h_t\tau_t}$')
    ax.plot(series.index, series['tau'] * A, color=IDAred, lw=1.6, label=rf'Long-term volatility $\sqrt{{\tau_t}}$, m = {m}')
    ax.set_ylim(0, 80)
    ax.set_ylabel('Annualised volatility (%)')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
    labs = [h[0] for h in HOR]
    xx = np.arange(len(labs))
    for k, col, name in (('gjr', Navy, 'GJR-GARCH'), ('mf2', IDAred, 'MF2-GARCH')):
        bx.plot(xx, [oos_full[l][k]['qlike_rel'] for l in labs], color=col, marker='o', ms=3.5, lw=1.3,
                label=f'{name}, 2010-2026')
        bx.plot(xx, [oos_pre[l][k]['qlike_rel'] for l in labs], color=col, marker='s', ms=3, lw=1.0, ls='--',
                label=f'{name}, 2010-2019')
    bx.axhline(1, color=Navy, lw=0.6, ls=':')
    bx.set_xticks(xx)
    bx.set_xticklabels(['1d', '1w'] + [f'M{k}' for k in range(1, 9)])
    bx.set_xlabel('Horizon: next day, next week, forward month M1-M8')
    bx.set_ylabel('QLIKE / QLIKE of the\nhistorical forecast')
    bx.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=2, frameon=False)
    plt.tight_layout()
    os.makedirs(CHART_DIR, exist_ok=True)
    for ext, kw in (('pdf', {}), ('png', dict(dpi=180))):
        plt.savefig(os.path.join(CHART_DIR, f'ch5_mf2_garch.{ext}'), bbox_inches='tight', transparent=True, **kw)
    plt.close()
    print('   saved ch5_mf2_garch')


def main():
    r = pct_returns('sp500', start=START, end=END)
    ppy = periods_per_year(r)
    print('S&P 500', r.index[0].date(), r.index[-1].date(), len(r), 'days')
    fs, series = full_sample(r)
    print('full sample: m =', fs['m'], 'BIC MF2', round(fs['bic_mf2'], 5), 'GJR', round(fs['bic_gjr'], 5))
    o = out_of_sample(r)
    print('out of sample: m =', o['m'])
    ev_full = evaluate(o)
    ev_pre = evaluate(o, end='2019-12-31')
    # the share of days on which the long-term component is above its mean (volatility cycles)
    tmean = fs['mf2']['lambda0'] / (1 - fs['mf2']['lam'])
    J = dict(fs, ppy=ppy, paper=PAPER, oos=dict(m=o['m'], start=str(o['idx'][o['i0']].date()), full=ev_full, pre=ev_pre,
                                                  window_days=int(o['i0']), step=STEP),
             tau_mean_vol=float(np.sqrt(tmean * ppy)),
             tau_max=dict(date=str(series['tau'].idxmax().date()), vol=float(series['tau'].max() * np.sqrt(ppy))),
             tau_min=dict(date=str(series['tau'].iloc[BURN:].idxmin().date()), vol=float(series['tau'].iloc[BURN:].min() * np.sqrt(ppy))),
             tau_last=float(series['tau'].iloc[-1] * np.sqrt(ppy)),
             horizons=[h[0] for h in HOR])
    # weekly series for the interactive page
    wk = series.iloc[BURN:].resample('W-FRI').last().dropna()
    J['weekly'] = dict(dates=[f'{d:%Y-%m-%d}' for d in wk.index], sigma=np.round(wk['sigma'].values * np.sqrt(ppy), 3).tolist(),
                       tau=np.round(wk['tau'].values * np.sqrt(ppy), 3).tolist())
    with open(OUT_JSON, 'w') as f:
        json.dump(J, f, indent=1, default=float)
    fig_case(series, ev_full, ev_pre, ppy, fs['m'])
    print(json.dumps({k: J[k] for k in ('m', 'gjr', 'mf2', 'mf2_se', 'bic_gjr', 'bic_mf2', 'kappa', 'tau_max', 'tau_min', 'tau_mean_vol')}, indent=1))
    for lab in J['horizons']:
        a, b = ev_full[lab], ev_pre[lab]
        print(f"{lab:5s} full: GJR {a['gjr']['qlike_rel']:.3f} MF2 {a['mf2']['qlike_rel']:.3f} DMql {a['dm_ql']:.2f} | "
              f"RMSE GJR {a['gjr']['rmse_rel']:.3f} MF2 {a['mf2']['rmse_rel']:.3f} DMse {a['dm_se']:.2f} || pre: "
              f"GJR {b['gjr']['qlike_rel']:.3f} MF2 {b['mf2']['qlike_rel']:.3f} DMql {b['dm_ql']:.2f} RMSE {b['gjr']['rmse_rel']:.3f}/{b['mf2']['rmse_rel']:.3f}  zeros {a['zero_days']}")



## Run

In [ ]:
main()

![ch5_mf2_garch](./ch5_mf2_garch.png)

Output: `ch5_mf2_garch.pdf`